In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
Version 8 — Fully 16-QAM Quantized VGG Network
═══════════════════════════════════════════════════════════════════════════════

This model implements a VGG-style CNN on CIFAR-10 using fully 16-QAM-
constrained computation for RF-aware neural inference. Unlike earlier
versions, RGB inputs are directly hard-quantized to a normalized 16-QAM
constellation without any learnable projection layer. All convolution
weights, linear weights, and hidden activations are constrained to the
same 16-QAM levels {(-3,-1,+1,+3)/√10} using Straight-Through Estimation
(STE) with latent float weights. Each layer maintains separate real and
imaginary weight tensors and performs RF-style magnitude recovery using
sqrt(real² + imag²). Compared to the QPSK-based Version 7, the higher
4-level quantization significantly improves representational capacity and
raises CIFAR-10 accuracy from 75.63% → 80.82%, while introducing higher
hardware complexity due to multi-bit arithmetic requirements.
═══════════════════════════════════════════════════════════════════════════════
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math

# ─────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# CONSTELLATION TABLES
# ─────────────────────────────────────────────
def get_qam_levels(n_levels: int) -> torch.Tensor:
    # Builds the 1-D axis of a square QAM constellation with n_levels points.
    # Generates n_per_axis odd integers centred at zero, then normalises so
    # RMS power equals 1.0. For 16-QAM yields [-3,-1,+1,+3]/√10.
    # Called at module load (for _QAM16_LEVELS) and in HardQAMInput.__init__.
    n_per_axis = int(math.sqrt(n_levels))
    assert n_per_axis * n_per_axis == n_levels, \
        f"n_levels must be a perfect square. Got {n_levels}"
    raw = torch.arange(-(n_per_axis - 1), n_per_axis, 2, dtype=torch.float32)
    rms = raw.pow(2).mean().sqrt()
    return raw / rms

# Pre-compute 16-QAM levels (used for weights and activations)
_QAM16_LEVELS = get_qam_levels(16)   # [-3,-1,+1,+3]/√10


# ─────────────────────────────────────────────
# 16-QAM WEIGHT/ACTIVATION QUANTIZER  (STE)
# ─────────────────────────────────────────────
class QAM16Quantize(torch.autograd.Function):
    # Snaps each element to the nearest 16-QAM level (4-way argmin) in the
    # forward pass and passes gradients straight through (STE) in the backward
    # pass. Replaces v7's binary torch.sign with a 4-level argmin, doubling
    # the representable values per weight at the cost of 2-bit multipliers.
    @staticmethod
    def forward(ctx, x: torch.Tensor) -> torch.Tensor:
        # Computes absolute distance to each of the 4 levels and returns
        # the level at the argmin index. Nothing saved to ctx — STE backward
        # returns grad unchanged without needing the input value.
        levels = _QAM16_LEVELS.to(x.device)           # (4,)
        dists  = (x.unsqueeze(-1) - levels).abs()     # (..., 4)
        idx    = dists.argmin(dim=-1)                 # (...,)
        return levels[idx]                            # (...,)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> torch.Tensor:
        # Straight-through estimator: upstream gradient passes unchanged,
        # bypassing the zero derivative of argmin so latent weights keep
        # receiving useful update signals throughout training.
        return grad_output.clone()

qam16_quantize = QAM16Quantize.apply


# ─────────────────────────────────────────────
# HARD 16-QAM INPUT QUANTIZER  (no STE needed)
# ─────────────────────────────────────────────
class HardQAMInput(nn.Module):
    # Hard 16-QAM quantization of normalized RGB pixels. Zero learnable
    # parameters. No STE needed — image tensors have requires_grad=False
    # so the argmin snap never blocks any gradient path.
    def __init__(self, n_levels: int = 16):
        # Registers constellation levels as a non-trainable buffer so they
        # move to the correct device with the model. Caches max_val for
        # use as the Tanh bounding scale in the forward pass.
        super().__init__()
        levels  = get_qam_levels(n_levels)
        max_val = levels.abs().max().item()
        self.register_buffer('levels', levels)
        self.max_val    = max_val
        self.n_levels   = n_levels
        self.n_per_axis = int(math.sqrt(n_levels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Tanh-bounds input to [-max_level, +max_level], then argmin-snaps
        # each element to the nearest constellation point. No STE wrapper
        # needed since pixel values carry no gradient.
        x = torch.tanh(x) * self.max_val
        dists  = (x.unsqueeze(-1) - self.levels.view(1,1,1,1,-1)).abs()
        idx    = dists.argmin(dim=-1)
        return self.levels[idx]   # no STE — input has no grad

    def diagnostics(self, x_raw: torch.Tensor) -> dict:
        # Runs forward in no-grad mode and returns sanity-check statistics:
        # unique level count (should equal n_per_axis), output range, mean,
        # and exact level values seen. Called once before training starts.
        with torch.no_grad():
            x_q = self.forward(x_raw)
        unique = x_q.unique()
        return {
            "n_unique_levels": unique.numel(),
            "expected_levels": self.n_per_axis,
            "min":  x_q.min().item(),
            "max":  x_q.max().item(),
            "mean": x_q.mean().item(),
            "levels_seen": [f"{v:.4f}" for v in unique.tolist()],
        }


# ─────────────────────────────────────────────
# DIAGNOSTICS HELPERS
# ─────────────────────────────────────────────
def sign_commitment(w: torch.Tensor) -> float:
    # Fraction of latent weights with |w| > 0.1 — proxy for how firmly each
    # weight has committed away from zero toward a 16-QAM level. Close to 1.0
    # means stable snapping; near 0.0 means weights are still undecided.
    return (w.abs() > 0.1).float().mean().item()


def qam16_snap_error(w_real: torch.Tensor, w_imag: torch.Tensor) -> float:
    # Mean absolute distance between each latent weight and its nearest 16-QAM
    # level, averaged over real and imaginary components. Small values mean
    # latent weights have converged close to the discrete grid.
    levels = _QAM16_LEVELS.to(w_real.device)

    def snap(w):
        dists = (w.unsqueeze(-1) - levels).abs()
        return levels[dists.argmin(dim=-1)]

    err_r = (w_real - snap(w_real)).abs().mean().item()
    err_i = (w_imag - snap(w_imag)).abs().mean().item()
    return (err_r + err_i) / 2.0


def sign_penalty(w_real: torch.Tensor, w_imag: torch.Tensor) -> torch.Tensor:
    # Bimodal regularizer: exp(-|w|) penalizes weights near zero, pushing
    # them toward a committed 16-QAM level. Works the same as in v7 —
    # the penalty peaks at zero regardless of constellation width.
    return (torch.exp(-w_real.abs()) + torch.exp(-w_imag.abs())).mean()


# ─────────────────────────────────────────────
# 16-QAM CONV LAYER
# ─────────────────────────────────────────────
class QAM16Conv2d(nn.Module):
    # Drop-in Conv2d replacement with 16-QAM-constrained latent weights.
    # Stores separate real/imag weight tensors, snaps both via qam16_quantize,
    # runs two convolutions, and returns complex magnitude sqrt(r²+i²).
    # Only change from v7's QPSKConv2d: 4-level argmin replaces 2-level sign,
    # and clip_weights uses a 16-QAM-appropriate ±1.5×max_level bound.
    def __init__(self, in_channels, out_channels, kernel_size,
                 padding=0, quantize_input=True):
        # Allocates real/imag latent weight tensors, stores padding and the
        # quantize_input flag, then calls _init_weights.
        super().__init__()
        self.quantize_input = quantize_input
        self.w_real = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.w_imag = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.padding = padding
        self._init_weights()

    def _init_weights(self):
        # Kaiming uniform scaled by 0.3 so initial weights sit near zero,
        # within the innermost 16-QAM levels, giving the sign-penalty room
        # to push weights outward without immediately saturating.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Optionally re-quantizes input activations to 16-QAM, then runs two
        # convolutions with snapped real/imag weights and returns the complex
        # magnitude. First layer sets quantize_input=False so hard-snapped
        # pixel values from HardQAMInput pass through unmodified.
        if self.quantize_input:
            x = qam16_quantize(x)
        w_q_real = qam16_quantize(self.w_real)
        w_q_imag = qam16_quantize(self.w_imag)
        out_real = F.conv2d(x, w_q_real, padding=self.padding)
        out_imag = F.conv2d(x, w_q_imag, padding=self.padding)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns sign_commitment and snap_error for real and imag weights.
        # Used in the periodic diagnostic logging pass to monitor convergence
        # of latent weights toward the discrete 16-QAM grid.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qam16_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty for this layer's weights,
        # to be scaled by λ_conv and summed into the total training loss.
        return sign_penalty(self.w_real, self.w_imag)

    def clip_weights(self):
        # Clamps latent weights to ±1.5×max_level after each optimizer step.
        # 16-QAM analogue of v7's BinaryConnect ±1 clipping — wider bound
        # leaves margin beyond the outermost level so gradients aren't zeroed.
        max_val = _QAM16_LEVELS.abs().max().item() * 1.5
        self.w_real.data.clamp_(-max_val, max_val)
        self.w_imag.data.clamp_(-max_val, max_val)


# ─────────────────────────────────────────────
# 16-QAM LINEAR LAYER
# ─────────────────────────────────────────────
class QAM16Linear(nn.Module):
    # FC analogue of QAM16Conv2d: real/imag latent weights quantized to
    # 16-QAM at every forward pass, returning complex magnitude of two
    # linear projections. Replaces float FC layers so the full network
    # stays 16-QAM constrained end-to-end.
    def __init__(self, in_features, out_features):
        # Allocates real/imag latent weight matrices and calls _init_weights.
        super().__init__()
        self.w_real = nn.Parameter(torch.Tensor(out_features, in_features))
        self.w_imag = nn.Parameter(torch.Tensor(out_features, in_features))
        self._init_weights()

    def _init_weights(self):
        # Kaiming uniform scaled by 0.3 — same strategy as QAM16Conv2d,
        # placing initial weights near zero so the sign-penalty is effective
        # from the first epoch.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Quantizes input activations to 16-QAM, applies two linear
        # projections with snapped real/imag weights, and returns the
        # element-wise complex magnitude of the results.
        x = qam16_quantize(x)
        w_q_real = qam16_quantize(self.w_real)
        w_q_imag = qam16_quantize(self.w_imag)
        out_real = F.linear(x, w_q_real)
        out_imag = F.linear(x, w_q_imag)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Same health metrics as QAM16Conv2d.diagnostics() so all 16-QAM
        # layers share a uniform monitoring interface during diagnostic logs.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qam16_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty for this layer's weights,
        # to be scaled by λ_fc and summed into the total training loss.
        return sign_penalty(self.w_real, self.w_imag)

    def clip_weights(self):
        # Same ±1.5×max_level clamping as QAM16Conv2d.clip_weights(),
        # applied to FC latent weights after every optimizer step.
        max_val = _QAM16_LEVELS.abs().max().item() * 1.5
        self.w_real.data.clamp_(-max_val, max_val)
        self.w_imag.data.clamp_(-max_val, max_val)


# ─────────────────────────────────────────────
# NETWORK
# ─────────────────────────────────────────────
class QAM16Net(nn.Module):
    # VGG-style backbone with hard 16-QAM inputs, 16-QAM weights/activations,
    # and a float linear head. Every internal weight is 4-level constrained —
    # the only upgrade from v7 is swapping QPSK (2-level) for 16-QAM (4-level)
    # throughout, at the hardware cost of 2-bit multipliers vs XNOR gates.
    def __init__(self, num_classes=10):
        # Builds hard 16-QAM input quantizer, three QAM16Conv2d blocks with
        # BatchNorm and MaxPool, two QAM16Linear layers with BatchNorm1d,
        # and a float nn.Linear classification head.
        super().__init__()

        self.input_quantizer = HardQAMInput(n_levels=16)

        # Block 1
        self.conv1 = QAM16Conv2d(3,   128, 3, padding=1, quantize_input=False)
        self.bn1   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.conv2 = QAM16Conv2d(128, 128, 3, padding=1, quantize_input=True)
        self.bn2   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv3 = QAM16Conv2d(128, 256, 3, padding=1, quantize_input=True)
        self.bn3   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.conv4 = QAM16Conv2d(256, 256, 3, padding=1, quantize_input=True)
        self.bn4   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3
        self.conv5 = QAM16Conv2d(256, 512, 3, padding=1, quantize_input=True)
        self.bn5   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.conv6 = QAM16Conv2d(512, 512, 3, padding=1, quantize_input=True)
        self.bn6   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Classifier
        self.fc1    = QAM16Linear(8192, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc2    = QAM16Linear(1024, 1024)
        self.bn_fc2 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc_out = nn.Linear(1024, num_classes)   # float head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Hard 16-QAM input quantization → three conv blocks (BN + MaxPool)
        # → flatten → two 16-QAM FC layers (BN) → float linear head.
        # Stateless call signature — no β, no STE residual, no teacher input.
        x = self.input_quantizer(x)

        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = self.pool1(x)

        x = self.bn3(self.conv3(x))
        x = self.bn4(self.conv4(x))
        x = self.pool2(x)

        x = self.bn5(self.conv5(x))
        x = self.bn6(self.conv6(x))
        x = self.pool3(x)

        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        x = self.bn_fc2(self.fc2(x))
        return self.fc_out(x)

    def qam16_layers(self):
        # Returns (name, layer) pairs for all six conv and both FC layers —
        # single iteration point for diagnostics, penalty, and weight clipping.
        return [
            ("conv1", self.conv1), ("conv2", self.conv2),
            ("conv3", self.conv3), ("conv4", self.conv4),
            ("conv5", self.conv5), ("conv6", self.conv6),
            ("fc1",   self.fc1),   ("fc2",   self.fc2),
        ]

    def get_all_diagnostics(self):
        # Collects diagnostics from every 16-QAM layer in one call, keyed by
        # layer name. Used in the periodic logging pass without touching
        # individual layers directly.
        return {name: layer.diagnostics() for name, layer in self.qam16_layers()}

    def total_sign_penalty(self, lambda_conv: float, lambda_fc: float) -> torch.Tensor:
        # Sums each layer's sign-penalty weighted by λ_conv or λ_fc.
        # The resulting scalar is added to CE loss at every training step
        # to push latent weights toward committed 16-QAM levels.
        penalty = torch.tensor(0.0, device=next(self.parameters()).device)
        for name, layer in self.qam16_layers():
            lam = lambda_fc if name.startswith("fc") else lambda_conv
            penalty = penalty + lam * layer.penalty()
        return penalty

    def clip_all_weights(self):
        # Calls clip_weights() on every 16-QAM layer after each optimizer step,
        # keeping all latent weights within ±1.5×max_level so none drift beyond
        # the valid constellation range.
        for _, layer in self.qam16_layers():
            layer.clip_weights()


# ─────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────
def get_cifar10_loaders(batch_size=128, num_workers=2):
    # Standard CIFAR-10 loaders: random crop + horizontal flip for training,
    # normalisation only for test. num_workers forced to 0 on MPS to avoid
    # multiprocessing issues; pin_memory enabled only for CUDA. No AutoAugment
    # or CutMix — same minimal augmentation as v7 to keep the comparison fair.
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    safe_workers = 0 if device.type == "mps" else num_workers
    train_set = torchvision.datasets.CIFAR10(
        root='./data', train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=test_transform)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=safe_workers, pin_memory=(device.type == "cuda"))
    test_loader  = torch.utils.data.DataLoader(
        test_set,  batch_size=batch_size, shuffle=False,
        num_workers=safe_workers, pin_memory=(device.type == "cuda"))
    return train_loader, test_loader


# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
def evaluate(model, loader):
    # Runs inference over a full DataLoader in eval mode (BN uses running
    # stats, no dropout) with gradients off. Returns top-1 accuracy as %.
    # 16-QAM quantizer runs inside each layer's forward — no float fallback.
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# ─────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, lambda_conv, lambda_fc,
                    log_grads=False):
    # One full pass over the training loader: computes CE + sign-penalty loss,
    # back-propagates, optionally logs per-layer gradient norms on batch 0 of
    # epoch 1 to detect dead weights, clips all latent weights after each step,
    # and returns epoch-averaged CE loss, penalty loss, and training accuracy.
    model.train()
    running_ce  = 0.0
    running_pen = 0.0
    correct = total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs  = model(images)
        ce_loss  = F.cross_entropy(outputs, labels)
        pen_loss = model.total_sign_penalty(lambda_conv, lambda_fc)
        loss     = ce_loss + pen_loss
        loss.backward()

        # Gradient check — epoch 1, batch 0
        if batch_idx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Batch 0]")
            for lname, layer in [("conv1", model.conv1), ("conv5", model.conv5),
                                  ("fc1",   model.fc1),   ("fc2",   model.fc2)]:
                gr = layer.w_real.grad
                gi = layer.w_imag.grad
                if gr is not None:
                    print(f"  {lname}: grad_real={gr.abs().mean().item():.6f}  "
                          f"grad_imag={gi.abs().mean().item():.6f}")
                else:
                    print(f"  {lname}: NO GRADIENT ← broken!")
            print()

        optimizer.step()

        # Weight clipping — keep latent weights near constellation range
        model.clip_all_weights()

        running_ce  += ce_loss.item()
        running_pen += pen_loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    return running_ce / n, running_pen / n, 100.0 * correct / total


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # Wires all components together. Hyperparameters deliberately match v7
    # (Adam, weight_decay=0, MultiStepLR at 100/150, 180 epochs) so the only
    # variable between runs is the quantizer order. Verifies input quantizer
    # before training, checkpoints best model, and prints per-layer diagnostics
    # every DIAG_EVERY epochs. Final summary shows the full v5–v8 comparison.
    EPOCHS        = 180
    BATCH_SIZE    = 128
    LR            = 1e-3
    LAMBDA_CONV   = 1e-4
    LAMBDA_FC     = 3e-4
    LR_MILESTONES = [100, 150]
    LR_GAMMA      = 0.1
    DIAG_EVERY    = 20

    levels_str = [f"{v:.4f}" for v in _QAM16_LEVELS.tolist()]

    print("=" * 70)
    print("All-16-QAM CNN on CIFAR-10  [v8]")
    print("Input        : Hard 16-QAM per channel (no projection)")
    print("Weights/Acts : 16-QAM constrained")
    print(f"16-QAM levels: {levels_str}")
    print(f"Scheduler    : MultiStepLR  milestones={LR_MILESTONES}  γ={LR_GAMMA}")
    print(f"Sign penalty : λ_conv={LAMBDA_CONV}  λ_fc={LAMBDA_FC}")
    print(f"Epochs       : {EPOCHS}  |  grad logging: epoch 1")
    print("=" * 70)

    train_loader, test_loader = get_cifar10_loaders(BATCH_SIZE)
    model     = QAM16Net(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=LR_MILESTONES, gamma=LR_GAMMA)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters : {total_params:,}")

    # Verify input quantizer
    sample_images = next(iter(test_loader))[0][:64].to(device)
    diag = model.input_quantizer.diagnostics(sample_images)
    print(f"\n[Input Quantizer Check]")
    print(f"  Unique levels : {diag['n_unique_levels']} "
          f"(expected {diag['expected_levels']}) "
          f"{'✓' if diag['n_unique_levels'] == diag['expected_levels'] else '✗ BUG'}")
    print(f"  Range         : [{diag['min']:.4f}, {diag['max']:.4f}]")
    print(f"  Levels seen   : {diag['levels_seen']}")
    print()

    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        ce_loss, pen_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer,
            LAMBDA_CONV, LAMBDA_FC,
            log_grads=(epoch == 1)
        )
        test_acc = evaluate(model, test_loader)
        scheduler.step()

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), "qam16_cifar10_v8_best.pth")

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch:3d}/{EPOCHS}]  "
              f"CE: {ce_loss:.4f}  Pen: {pen_loss:.4f}  "
              f"Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  "
              f"Best: {best_acc:.2f}%  LR: {current_lr:.2e}")

        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}]")
            for lname, d in model.get_all_diagnostics().items():
                print(f"  {lname:6s}: "
                      f"commit_r={d['sign_commit_real']:.3f}  "
                      f"commit_i={d['sign_commit_imag']:.3f}  "
                      f"snap_err={d['snap_error']:.4f}")
            print()

    print(f"\nTraining complete. Best test accuracy: {best_acc:.2f}%")
    print(f"Model saved to: qam16_cifar10_v8_best.pth")
    print(f"\nFull comparison:")
    print(f"  v5  soft 16-QAM input + QPSK weights : 86.86%")
    print(f"  v6  soft QPSK  input + QPSK weights  : 86.81%")
    print(f"  v7  hard 16-QAM input + QPSK weights : 75.63%")
    print(f"  v8  hard 16-QAM input + 16-QAM weights: {best_acc:.2f}%  ← this run")


if __name__ == "__main__":
    main()